# Crawling detik.com — Finance & Olahraga (Fast + Colab-style Output)

**Target:** 2 tema × 100 berita = 200 artikel  
**Sumber:** `finance.detik.com` & `sport.detik.com`  
**Output:** `detik_berita.csv` → kolom `id, isi_berita, tema` (A=id, B=isi, C=tema, sep=; Excel-friendly)  
**Teknik:** `requests + BeautifulSoup` untuk discovery link `/d-\d+/`, `trafilatura` untuk extract, `tqdm` + `ThreadPool`

## Requirements

Install di venv `PPWA` (sudah termasuk `tqdm` untuk progress bar Colab-style):

```bash
pip install trafilatura pandas requests beautifulsoup4 lxml tqdm ipykernel
python -m ipykernel install --user --name ppwa --display-name "Python (ppwa)"
```

| Library | Fungsi |
|---|---|
| `trafilatura` | `fetch_url` + `extract(include_comments=False)` — ekstraksi isi bersih |
| `requests` + `beautifulsoup4/lxml` | fetch indeks & discovery link `/d-\d+/` |
| `tqdm` | progress bar live kayak Google Colab |
| `pandas` | DataFrame, `value_counts`, preview & save CSV `utf-8-sig` |
| `ipykernel` | Jupyter kernel untuk VS Code/Jupyter |

**Wajib:** pilih kernel **Python (ppwa)** kanan atas VS Code sebelum Run.

In [57]:
# cek versi library
import trafilatura, pandas as pd
print(f"trafilatura {trafilatura.__version__} | pandas {pd.__version__}")
import requests, bs4, lxml
print(f"requests {requests.__version__} | bs4 {bs4.__version__} | lxml {lxml.__version__}")
import tqdm
print(f"tqdm {tqdm.__version__} OK")

trafilatura 2.2.0 | pandas 3.0.5
requests 2.34.2 | bs4 4.15.0 | lxml 6.1.3
tqdm 4.70.0 OK


In [58]:
import requests
from bs4 import BeautifulSoup
import trafilatura
import pandas as pd
import re, time
from tqdm.auto import tqdm  # progress bar Colab-style
from concurrent.futures import ThreadPoolExecutor, as_completed

print("imports OK - tqdm + ThreadPool ready", flush=True)

imports OK - tqdm + ThreadPool ready


## Konfigurasi 2 Tema

In [59]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "id-ID,id;q=0.9,en;q=0.8",
}

TOPICS = [
    {"tema": "finance",  "url": "https://finance.detik.com/"},
    {"tema": "olahraga", "url": "https://sport.detik.com/"},
]

MAX_PER_TEMA = 100
OUTPUT_CSV = "detik_berita.csv"
pd.DataFrame(TOPICS)

,tema,url
0,finance,https://finance.detik.com/
1,olahraga,https://sport.detik.com/


## Fungsi — Discovery link & Extract via trafilatura

In [60]:
def fetch(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        return r.text
    except requests.RequestException as e:
        print(f"[ERROR] fetch {url}: {e}", flush=True)
        return None

def get_article_links(html, base_url):
    soup = BeautifulSoup(html, "lxml")
    seen, links = set(), []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if re.search(r"/d-\d+/", href) and href not in seen:
            if not href.startswith("http"):
                href = base_url.rstrip("/") + "/" + href.lstrip("/")
            if "detik.com" in href:
                seen.add(href)
                links.append(href)
    return links

def scrape_article_trafilatura(url):
    downloaded = trafilatura.fetch_url(url)
    if not downloaded:
        downloaded = fetch(url)
        if not downloaded:
            return None
    text = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
    if not text or len(text.strip()) < 100:
        return None
    # FIX: pakai \s+ yang benar + hapus newline/; agar CSV Excel 1 baris = 1 artikel, 1 cell = 1 kolom
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace(";", ",").replace("\r", " ").replace("\n", " ")
    soup = BeautifulSoup(downloaded, "lxml")
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else url
    return {"judul": title, "url": url, "isi_berita": text}

### Test 1 URL (sanity check)

In [61]:
test_url = "https://finance.detik.com/moneter/d-8652973/muncul-isu-gaji-asn-dipindah-ke-bank-bumn-purbaya-buka-suara"
art = scrape_article_trafilatura(test_url)
if art:
    print(art["judul"][:80], flush=True)
    print(f"panjang isi: {len(art['isi_berita'])} char", flush=True)
    print(art["isi_berita"][:400] + "...", flush=True)
else:
    print("extract gagal", flush=True)

Muncul Isu Gaji ASN Dipindah ke Bank BUMN, Purbaya Buka Suara
panjang isi: 1881 char
Menteri Keuangan Purbaya Yudhi Sadewa buka suara merespons isu pemindahan payroll atau pembayaran gaji aparatur sipil negara (ASN) pemerintah daerah (Pemda) ke bank BUMN. Purbaya menegaskan tidak ada kebijakan dari pemerintah pusat terkait pemindahan tersebut. Isu pengalihan pembayaran gaji ASN dari Bank Pembangunan Daerah (BPD) ke bank BUMN sebelumnya muncul dalam Rapat Dengar Pendapat Umum (RDPU...


## Main — Crawling 100 berita per tema (FAST + Colab Output)

Target `100/100` per tema → total `200` artikel. `ThreadPoolExecutor(max_workers=5)` + `tqdm` + `OK` tetap. Crawl `/indeks?page=2..15` sampai 400 kandidat.

In [62]:
all_data = []
for topic in TOPICS:
    print(f"\n=== {topic['tema'].upper()} | {topic['url']} ===", flush=True)
    html = fetch(topic["url"])
    if not html:
        continue
    links = get_article_links(html, topic["url"])
    print(f"  Ditemukan {len(links)} link", flush=True)
    candidates = links[:MAX_PER_TEMA*4]
    # target 100 -> butuh ~400 kandidat, crawl indeks 2..15
    if len(candidates) < MAX_PER_TEMA*3:
        for page in range(2, 16):
            for extra_url in [topic["url"].rstrip("/") + f"/indeks?page={page}", topic["url"] + f"?page={page}"]:
                extra_html = fetch(extra_url)
                if extra_html:
                    extra_links = get_article_links(extra_html, topic["url"])
                    new = 0
                    for l in extra_links:
                        if l not in candidates:
                            candidates.append(l)
                            new += 1
                    if new > 0:
                        print(f"  + halaman {page} ({extra_url[-20:]}): +{new} -> total {len(candidates)}", flush=True)
                        break
            if len(candidates) >= MAX_PER_TEMA*4:
                break
        print(f"  Setelah indeks tambahan: {len(candidates)} kandidat", flush=True)
    
    scraped = []
    with ThreadPoolExecutor(max_workers=5) as ex:
        futures = {ex.submit(scrape_article_trafilatura, link): link for link in candidates}
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"{topic['tema']}"):
            link = futures[f]
            try:
                art = f.result()
            except Exception as e:
                print(f"    ERROR {link[-40:]}: {e}", flush=True)
                continue
            if art:
                art["tema"] = topic["tema"]
                scraped.append(art)
                print(f"    OK | {art['judul'][:55]}... ({len(art['isi_berita'])} char)", flush=True)
                if len(scraped) >= MAX_PER_TEMA:
                    for fut in futures:
                        fut.cancel()
                    break
            else:
                print(f"    SKIP ...{link[-50:]}", flush=True)
    
    scraped = scraped[:MAX_PER_TEMA]
    print(f"  -> Terkumpul {len(scraped)}/{MAX_PER_TEMA} untuk {topic['tema']}", flush=True)
    all_data.extend(scraped)
    time.sleep(0.3)

# deduplicate by url
seen = set()
unique = []
for a in all_data:
    if a["url"] not in seen:
        seen.add(a["url"])
        unique.append(a)
all_data = unique
print(f"\nTotal unik: {len(all_data)} artikel", flush=True)


=== FINANCE | https://finance.detik.com/ ===
  Ditemukan 46 link
  + halaman 2 (ik.com/indeks?page=2): +3 -> total 49
  + halaman 3 (ik.com/indeks?page=3): +20 -> total 69
  + halaman 4 (ik.com/indeks?page=4): +20 -> total 89
  + halaman 5 (ik.com/indeks?page=5): +20 -> total 109
  + halaman 6 (ik.com/indeks?page=6): +17 -> total 126
  + halaman 7 (ik.com/indeks?page=7): +20 -> total 146
  + halaman 8 (ik.com/indeks?page=8): +19 -> total 165
  + halaman 9 (ik.com/indeks?page=9): +15 -> total 180
  + halaman 10 (k.com/indeks?page=10): +19 -> total 199
  + halaman 11 (k.com/indeks?page=11): +20 -> total 219
  + halaman 12 (k.com/indeks?page=12): +20 -> total 239
  + halaman 13 (k.com/indeks?page=13): +19 -> total 258
  + halaman 14 (k.com/indeks?page=14): +18 -> total 276
  + halaman 15 (k.com/indeks?page=15): +17 -> total 293
  Setelah indeks tambahan: 293 kandidat


finance:   0%|          | 0/293 [00:00<?, ?it/s]

    OK | Bandara Soetta-Husein Dibuka Lagi, Radin Inten Masih Di... (2221 char)


finance:   0%|          | 1/293 [00:00<02:01,  2.40it/s]

    OK | Bandara Soetta-Husein Siap Beroperasi Lagi Usai Dibuka... (2702 char)


...

    OK | Men's World Tennis Championship: Gunawan Tembus Babak U... (1649 char)
    OK | Duel 300 Km/Jam Marc Marquez Vs Bezzecchi di Aragon: Mo... (1749 char)


olahraga:  30%|███       | 99/325 [00:20<00:55,  4.10it/s]

    OK | Jawa Timur Juara Kejurnas Panahan Tradisional 2026... (1709 char)


olahraga:  30%|███       | 99/325 [00:20<00:45,  4.93it/s]


  -> Terkumpul 100/100 untuk olahraga

Total unik: 200 artikel


## Result — CSV + Preview (3 kolom Excel-friendly: A=id, B=isi_berita, C=tema)

In [63]:
# Result CSV - 3 kolom Excel-friendly: id | isi_berita | tema (A=id, B=isi, C=tema)
import csv
# buat DataFrame dengan urutan sesuai request: A=id, B=isi_berita, C=tema
df = pd.DataFrame([{"id": idx, "isi_berita": a["isi_berita"].replace(";", ",").replace("\n"," ").replace("\r"," "), "tema": a["tema"]} for idx, a in enumerate(all_data, 1)])
df = df[["id", "isi_berita", "tema"]]  # urutan: A=id, B=isi_berita, C=tema
assert list(df.columns) == ["id", "isi_berita", "tema"], "Kolom harus 3: id, isi_berita, tema"
assert df.shape[1] == 3, "Jumlah kolom harus 3"
# cek isi tidak ada newline/; yang bikin Excel pindah cell/baris
assert not df["isi_berita"].str.contains(";").any(), "isi_berita masih ada ;"
# simpan dengan sep=; (Excel Indonesia) dan quote minimal agar 1 baris = 1 artikel
try:
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig", sep=";", quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
    print(f"Saved -> {OUTPUT_CSV} : {df.shape[0]} rows x {df.shape[1]} cols (sep=';')", flush=True)
except PermissionError:
    alt = OUTPUT_CSV.replace(".csv", "_new.csv")
    df.to_csv(alt, index=False, encoding="utf-8-sig", sep=";", quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
    print(f"[WARN] {OUTPUT_CSV} terkunci -> saved ke {alt} : {df.shape[0]} rows", flush=True)
print(f"Jumlah kolom: {df.shape[1]}", flush=True)
print(f"Daftar kolom (3 kolom): {df.columns.tolist()} -> A=id, B=isi_berita, C=tema", flush=True)
print(f"  - A:id         : nomor urut (1..n)", flush=True)
print(f"  - B:isi_berita : teks 1 baris (tanpa newline/;)", flush=True)
print(f"  - C:tema       : finance / olahraga", flush=True)
print(f"\nTotal artikel: {len(df)}", flush=True)
display(df.head(3))
print("\nDistribusi tema (tabel):", flush=True)
tabel_tema = df["tema"].value_counts().to_frame("jumlah")
tabel_tema = tabel_tema.reindex([t for t in ["finance", "olahraga"] if t in tabel_tema.index])
display(tabel_tema)
for tema, jumlah in tabel_tema["jumlah"].items():
    print(f"  - {tema}: {jumlah} artikel", flush=True)


Saved -> detik_berita.csv : 200 rows x 3 cols (sep=';')
Jumlah kolom: 3
Daftar kolom (3 kolom): ['id', 'isi_berita', 'tema'] -> A=id, B=isi_berita, C=tema
  - A:id         : nomor urut (1..n)
  - B:isi_berita : teks 1 baris (tanpa newline/;)
  - C:tema       : finance / olahraga

Total artikel: 200


,id,isi_berita,tema
0,1,Penutupan operasional Bandara Soekarno-Hatta b...,finance
1,2,InJourney Airports menjamin kesiapan operasion...,finance
2,3,Menteri Keuangan (Menkeu) Purbaya Yudhi Sadewa...,finance



Distribusi tema (tabel):


,jumlah
tema,
finance,100
olahraga,100


  - finance: 100 artikel
  - olahraga: 100 artikel
